# NLP Group 7 Project P2

## Load the dataset

In [15]:
from datasets import load_dataset
from openai import OpenAI
from dotenv import load_dotenv
import pandas as pd
import os
import json

## Load dataset

In [16]:
dataset = load_dataset("MathArena/final_answer_comps", split="train")
df = dataset.to_pandas()

## General info and null values

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 139 entries, 0 to 138
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   problem_idx   139 non-null    int64 
 1   answer        139 non-null    object
 2   problem_type  130 non-null    object
 3   problem       139 non-null    object
 4   competition   139 non-null    object
 5   source        9 non-null      object
dtypes: int64(1), object(5)
memory usage: 6.6+ KB


In [18]:
print("Dataset Head:\n", df.head(), sep="", end="\n" + "="*50 + "\n")
print("Dataset Columns:\n", df.columns, sep="", end="\n" + "="*50 + "\n")
print("Dataset Shape:\n", df.shape, sep="", end="\n" + "="*50 + "\n")

Dataset Head:
   problem_idx answer                    problem_type  \
0            1     70                 [Number Theory]   
1            2    588                      [Geometry]   
2            3     16                 [Combinatorics]   
3            4    117                       [Algebra]   
4            5    279  [Combinatorics, Number Theory]   

                                             problem     competition source  
0  Find the sum of all integer bases $b>9$ for wh...  aime/aime_2025   None  
1  On $\triangle ABC$ points $A, D, E$, and $B$ l...  aime/aime_2025   None  
2  The 9 members of a baseball team went to an ic...  aime/aime_2025   None  
3  Find the number of ordered pairs $(x,y)$, wher...  aime/aime_2025   None  
4  There are $8!= 40320$ eight-digit positive int...  aime/aime_2025   None  
Dataset Columns:
Index(['problem_idx', 'answer', 'problem_type', 'problem', 'competition',
       'source'],
      dtype='object')
Dataset Shape:
(139, 6)


## Communication with the model

In [11]:
load_dotenv()

ENDPOINT = "https://nlp-pcaf.services.ai.azure.com/openai/v1/"
MODEL_NAME = "DeepSeek-V3-0324"
DEPLOYMENT_NAME = "DeepSeek-V3-0324"

api_key = os.getenv("API_KEY")

client = OpenAI(
    base_url=f"{ENDPOINT}",
    api_key=api_key
)

completion = client.chat.completions.create(
    model=DEPLOYMENT_NAME,
    messages=[
        {
            "role": "user",
            "content": "What is the capital of France?",
        }
    ],
)

print(completion.choices[0].message)

ChatCompletionMessage(content='The capital of France is **Paris**. It is one of the most famous and visited cities in the world, known for landmarks like the Eiffel Tower, the Louvre Museum, and Notre-Dame Cathedral.  \n\nWould you like information on anything specific about Paris? 😊', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)


In [12]:
def get_llm_response(role_prompt, user_input, temperature=0.0):
    """Sends a request to the Azure LLM with a System Role Prompt."""
    messages = [
        {"role": "system", "content": role_prompt},
        {"role": "user", "content": user_input}
    ]
    
    # Configure the response format for JSON
    response_format = {"type": "text"}
    if "JSON" in role_prompt or "JSON" in user_input:
         response_format = {"type": "json_object"}
    
    try:
        response = client.chat.completions.create(
            model=DEPLOYMENT_NAME,
            messages=messages,
            temperature=temperature,
            response_format=response_format
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"LLM API Error: {e}"

## Role Prompts

In [30]:
VERIFIER_JSON_SCHEMA = """
{
  "valid": <boolean: true if final answer is correct, false otherwise>,
  "error_category": <string: one of 'CALCULATION_ERROR', 'CONCEPTUAL_FLAW', 'LOGIC_OMISSION', 'NONE'>,
  "critique_summary": <string: a brief, actionable explanation of the error>
}
"""

PROVER_ROLE = (
    "You are the PROVER, an expert mathematician. Your goal is to generate a detailed, "
    "step-by-step Chain-of-Thought solution to the problem. Start with the initial setup. "
    "If given a CRITIQUE, strictly follow it to correct your next attempt."
)

VERIFIER_ROLE = (
    "You are the VERIFIER, a hyper-critical logic machine. Your sole task is to carefully analyze the "
    "provided solution and output a JSON object strictly adhering to the specified schema. "
    f"Analyze the solution for correctness and logic. Output strictly valid JSON only. No other comments, just a valid JSON object matching this schema: {VERIFIER_JSON_SCHEMA}"
    "Error categories are: 'CALCULATION_ERROR', 'CONCEPTUAL_FLAW', 'LOGIC_OMISSION', 'NONE'. "
)

# Example Problem
PROBLEM = (
    "A rectangular garden has sides in the ratio 4:3. If the area of the garden is 300 m^2, "
    "what is the length of the fence needed to enclose it? Provide your final answer as an integer."
)

# Example of a Flawed Solution (Prover's first attempt for PoC)
# This simulates the Prover making a calculation error: 2*(20+15) = 70, but the Prover will output 90.
FLAWED_SOLUTION_S1 = """
Solution:
1. Let the length be 4x and the width be 3x.
2. Area: (4x)(3x) = 12x^2.
3. 12x^2 = 300, so x^2 = 25, and x = 5.
4. Sides are 4(5)=20m and 3(5)=15m.
5. Perimeter P = 2(20 + 15) = 2(45) = 90m.
Final Answer: 90
"""

## PoC run

In [31]:
verifier_input = (
    f"Problem: {PROBLEM}\n\n"
    f"Solution to Critique:\n{FLAWED_SOLUTION_S1}"
)

verifier_raw_output = get_llm_response(VERIFIER_ROLE, verifier_input, temperature=0.0)
verifier_raw_output = verifier_raw_output.replace("```json", "")
verifier_raw_output = verifier_raw_output.replace("```", "")
verifier_raw_output = verifier_raw_output.strip()

try:
    # JSON parsing
    verifier_json = json.loads(verifier_raw_output)
    print("JSON Parsing Successful. Verifier Output:")
    print(json.dumps(verifier_json, indent=2))
    
    # Check if the critique is valid and get correction signal
    if verifier_json.get('valid') == False:
        error_cat = verifier_json.get('error_category', 'UNKNOWN_ERROR')
        critique = verifier_json.get('critique_summary', 'No summary provided.')
        
        print("\nSolution is invalid. Preparing Planner's Correction...")
        
        # Planner logic
        planner_correction_hint = (
            f"CRITIQUE: The Verifier identified a **{error_cat}** at the final step. "
            f"Specifically: **{critique}**. You must rigorously re-examine your final calculation."
        )

        # Next Prover input
        prover_input_s2 = (
            f"ORIGINAL PROBLEM: {PROBLEM}\n\n"
            f"PREVIOUS FAILED ATTEMPT:\n{FLAWED_SOLUTION_S1}\n\n"
            f"PLANNER'S CORRECTION HINT:\n{planner_correction_hint}\n\n"
            f"--- GENERATE CORRECTED SOLUTION (ATTEMPT S2) ---"
        )
        
    else:
        print("\nSolution passed. Loop terminates.")
        prover_input_s2 = None
        
except json.JSONDecodeError as e:
    print(f"JSON Failure: Error: {e}")
    prover_input_s2 = None # Fail the loop

# PCAF iteration 2 (Prover Correction)
if prover_input_s2:
    print("\n--- 2. PROVER CORRECTION (Targeted Generation Proof) ---")
    
    # Send the corrected prompt to the single LLM instance
    prover_corrected_output = get_llm_response(PROVER_ROLE, prover_input_s2, temperature=0.0)
    
    print("Prover's Corrected Solution (S2):")
    print(prover_corrected_output)
    
    # We can run the verifier again and have multiple iterations but this is for PoC.
    print("\n[PoC Complete] The system successfully ran one full corrective loop.")
    print("The final correctness of S2 would be checked against the MathArena gold standard.")

JSON Parsing Successful. Verifier Output:
{
  "valid": true,
  "error_category": "NONE",
  "critique_summary": "The solution correctly follows the steps to determine the perimeter of the garden, with accurate calculations or conceptual errors."
}

Solution passed. Loop terminates.
